<div align="center">

<a href="https://colab.research.google.com/github/utkukose/rd-project-management-NB-lecture/blob/main/exams/final/FIN2_team_flow_simulation.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Final Capstone: Managing Research Projects

## Project 2: An agent-based simulation of a research team: Scrum, Kanban and multitasking

**R&D and Project Management in Computer Science (11117BLG002)**<br>Prof. Dr. Utku Kose<br>Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

*This course is updated in line with current developments in the field. Last update: September 2026.*

</div>

## The project

A team of four researchers works through a backlog of uncertain tasks that generates new work as it is completed. The example compares pushing all work into progress, Scrum sprints whose commitment follows the effort delivered in recent sprints and Kanban with three limits on work in progress, with a cost for switching between tasks [1, 2, 3].

**Required extensions.** Add milestones with deadlines from a grant agreement, heterogeneous skills and absences, and measure lateness against milestones as well as throughput and cycle time. Validate the model qualitatively against Little's law and the literature on agile research projects, and recommend a working method for a funded project [4, 5].

## What this example implements

The notebook simulates 200 working days for five policies, reports throughput, mean and 85th percentile cycle time and average work in progress, and shows how the cost of context switching changes throughput.

Run the cells in order. The example is a baseline: the capstone extends it, evaluates the extensions and reports the results.

In [ ]:
# Setup: common libraries and the self-check helper used in the exercises.
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SEED = 2026

def check(name, value, expected, tol=1e-6):
    """Compare an exercise answer with the reference value and print feedback."""
    if value is None:
        print(f"[{name}] Not attempted yet. Replace None with your answer and run the cell again.")
        return
    try:
        ok = bool(np.allclose(np.asarray(value, dtype=float), np.asarray(expected, dtype=float), atol=tol))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"[{name}] " + ("Correct." if ok else "Not yet. Revisit the explanation above and try again."))

## 1. Five ways to organise the work

In [ ]:
import pandas as pd

def simulate_team(policy, days=200, team=4, switch_cost=0.15, seed=SEED):
    rng = np.random.default_rng(seed)
    new_item = lambda: dict(size=float(rng.lognormal(1.2, 0.6)), start=None, left=None, end=None)
    backlog = [new_item() for _ in range(150)]
    wip, done, velocity, delivered, wip_trace = [], [], [], 0.0, []
    for day in range(days):
        if policy[0] == "scrum" and day % 10 == 0:
            cap = np.mean(velocity[-3:]) if velocity else 1.0 * team * 10
            load = sum(it["left"] for it in wip)
            for it in list(backlog):
                if load >= cap and len(wip) >= team:
                    break
                if load + it["size"] <= cap or len(wip) < team:
                    backlog.remove(it); it["start"], it["left"] = day, it["size"]; wip.append(it); load += it["size"]
        elif policy[0] in ("kanban", "push"):
            limit = policy[1] if policy[0] == "kanban" else 40
            while backlog and len(wip) < limit:
                it = backlog.pop(0); it["start"], it["left"] = day, it["size"]; wip.append(it)
        wip_trace.append(len(wip))
        for person in range(team):
            items = wip[person::team]
            if items:
                eff = 1.0 / (1 + switch_cost * (len(items) - 1))
                delivered += eff
                for it in items:
                    it["left"] -= eff / len(items)
        finished = [it for it in wip if it["left"] <= 1e-9]
        for it in finished:
            it["end"] = day + 1; done.append(it)
            if rng.random() < 0.25:
                backlog.append(new_item())
        wip = [it for it in wip if it["left"] > 1e-9]
        if policy[0] == "scrum" and day % 10 == 9:
            velocity.append(delivered); delivered = 0.0
    ct = np.array([it["end"] - it["start"] for it in done])
    return dict(throughput=len(done) / days, mean_cycle_time=ct.mean(), p85_cycle_time=np.percentile(ct, 85), mean_wip=np.mean(wip_trace))

POLICIES = [("push",), ("scrum",), ("kanban", 2), ("kanban", 4), ("kanban", 8)]
table = pd.DataFrame({" ".join(map(str, p)): simulate_team(p) for p in POLICIES}).T
table.round(2)

## 2. The cost of multitasking

In [ ]:
costs = [0.0, 0.1, 0.2, 0.3]
for p in [("push",), ("kanban", 4)]:
    tp = [simulate_team(p, switch_cost=c)["throughput"] for c in costs]
    plt.plot(costs, tp, "o-", label=" ".join(map(str, p)))
plt.xlabel("cost of context switching"); plt.ylabel("items completed per day"); plt.legend(); plt.title("Multitasking and throughput"); plt.show()

## Report and submission

Both tracks follow the course report template. The report states a question, justifies the approach, presents evidence from the literature or from the simulation with figures, discusses limitations and ends with recommendations for managing research projects. Sources are cited in square brackets.

This capstone also supports self-learning and can be completed at any pace. When the course is taught actively in a semester, the final capstone is sent by e-mail to utkukose@sdu.edu.tr or utkukose@gmail.com no later than 23:53 (Türkiye time) on the last Sunday of Week 14, with the report and all code files attached or linked.

## References

[1] Schwaber, K., & Sutherland, J. (2020). *The Scrum Guide: The Definitive Guide to Scrum: The Rules of the Game*. <https://scrumguides.org>

[2] Anderson, D. J. (2010). *Kanban: Successful Evolutionary Change for Your Technology Business*. Blue Hole Press.

[3] Little, J. D. C. (1961). A proof for the queuing formula: L = λW. *Operations Research*, 9(3), 383-387. <https://doi.org/10.1287/opre.9.3.383>

[4] Marchesi, M., Mannaro, K., Uras, S., & Locci, M. (2007). Distributed Scrum in research project management. In *Agile Processes in Software Engineering and Extreme Programming (XP 2007), Lecture Notes in Computer Science 4536* (pp. 240-244). <https://doi.org/10.1007/978-3-540-73101-6_45>

[5] Dingsøyr, T., Nerur, S., Balijepally, V., & Moe, N. B. (2012). A decade of agile methodologies: Towards explaining agile software development. *Journal of Systems and Software*, 85(6), 1213-1221. <https://doi.org/10.1016/j.jss.2012.02.033>